# IMPORT

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# DEFINE FOLDERS

In [ ]:
BASE_FOLDER = Path("../data")
STATION_FEATURE_FOLDER = BASE_FOLDER / "02_features" / "station_features"
DATASET_INPUT_FOLDER = BASE_FOLDER / "01_clean" / "datasets"

folders = [DATASET_INPUT_FOLDER, STATION_FEATURE_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA

In [8]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/final_clean_split_data.csv")

print("(Rows, Columns):", df.shape)
print("Stations:", df["station_code"].nunique())
print(df.columns)

(Rows, Columns): (6710838, 20)
Stations: 1417
Index(['timestamp', 'station_code', 'latitude', 'longitude', 'capacity',
       'total_bikes_now', 'total_bikes_taken', 'temperature', 'wind_speed',
       'wind_direction', 'weather_code', 'is_day', 'hour', 'day_of_week',
       'is_weekend', 'sin_hour', 'cos_hour', 'total_bikes_lag1',
       'total_bikes_lag2', 'split'],
      dtype='object')


# NORMALIZE USAGE

In [9]:
df["usage_ratio"] = df["total_bikes_now"] / df["capacity"]
df["usage_ratio"] = df["usage_ratio"].clip(0, 1)
# We normalize data as there are numerous different values for capacity.
# Normalizing it makes it possible to easily compare it to other stations.

# DEFINE FEATURE EXTRACTION PER STATION

In [10]:
def extract_station_features(group):
    features = {}

    # Basic usage(mean, std, min and max)
    features["mean_usage"] = group["usage_ratio"].mean()
    features["std_usage"] = group["usage_ratio"].std()
    features["min_usage"] = group["usage_ratio"].min()
    features["max_usage"] = group["usage_ratio"].max()

    # Temporal behavior(the difference between the usage on weekdays and weekends)
    features["mean_weekend"] = group[group["is_weekend"] == 1]["usage_ratio"].mean()
    features["mean_weekday"] = group[group["is_weekend"] == 0]["usage_ratio"].mean()

    features["day_usage"] = group[group["is_day"] == 1]["usage_ratio"].mean()
    features["night_usage"] = group[group["is_day"] == 0]["usage_ratio"].mean()

    # Daily amplitude(how much it varies during the day)
    hourly_mean = group.groupby("hour")["usage_ratio"].mean()
    features["daily_amplitude"] = hourly_mean.max() - hourly_mean.min()

    # Weather correlation
    for col in [
        "temperature",
        "wind_speed",
        "wind_direction",
        "weather_code"
    ]:
        if group[col].nunique() > 1:
            features[f"corr_{col}"] = group["usage_ratio"].corr(group[col])
        else:
            features[f"corr_{col}"] = 0.0

    # Station info (static/spatial)
    features["latitude"] = group["latitude"].iloc[0]
    features["longitude"] = group["longitude"].iloc[0]
    features["capacity"] = group["capacity"].iloc[-1] # To get the latest capacity

    # NO BIKE LAG
    # Lag features are time-dependent series and should not be aggregated at station level.
    # They are computed later during model training.
    # features["total_bikes_lag1"] = group["total_bikes_lag1"]
    # features["total_bikes_lag2"] = group["total_bikes_lag2"]

    return pd.Series(features)


# BUILD STATION FEATURE TABLE

In [11]:
station_features = (
    df.groupby("station_code", sort=False, group_keys=False)
    .apply(extract_station_features)
    .reset_index()
)

print("Station feature table shape:", station_features.shape)
station_features.head()
# Now we have features for each station.

Station feature table shape: (1417, 17)


C:\Users\emirh\AppData\Local\Temp\ipykernel_163892\2997561631.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(extract_station_features)


,station_code,mean_usage,std_usage,min_usage,max_usage,mean_weekend,mean_weekday,day_usage,night_usage,daily_amplitude,corr_temperature,corr_wind_speed,corr_wind_direction,corr_weather_code,latitude,longitude,capacity
0,1001,0.560011,0.281112,0.0,1.000000,0.513568,0.578413,0.653576,0.506016,0.379505,0.018291,0.206319,0.181526,-0.072267,48.857059,2.341798,17.0
1,1002,0.519967,0.300805,0.0,1.000000,0.448492,0.548287,0.617833,0.463489,0.557531,0.141096,0.191486,0.150732,-0.042812,48.857993,2.346917,22.0
2,1006,0.465307,0.228911,0.0,0.939394,0.457781,0.468291,0.429855,0.485780,0.397882,0.054264,0.183888,0.047263,-0.092844,48.862630,2.349807,33.0
3,1007,0.583198,0.268896,0.0,1.000000,0.537458,0.601332,0.715414,0.506847,0.529762,0.286513,0.299294,0.097743,-0.037711,48.863589,2.347779,16.0
4,1008,0.504804,0.300634,0.0,1.000000,0.478562,0.515202,0.630725,0.432137,0.564778,0.243769,0.270273,0.148330,-0.050365,48.863627,2.342624,32.0


# CLEAN FEATURE TABLE

In [12]:
stations_before = set(station_features["station_code"])

print(f"Before cleaning: {station_features.shape}")
station_features = station_features.replace([np.inf, -np.inf], np.nan)
station_features = station_features.dropna()
print(f"After cleaning: {station_features.shape}")

stations_after = set(station_features["station_code"])

dropped_stations = stations_before - stations_after

print(f"Dropped stations: {len(dropped_stations)}")
print(dropped_stations)
# Dropping stations in case we have invalid values(inf/NaN) that we have missed(we did not).

Before cleaning: (1417, 17)
After cleaning: (1417, 17)
Dropped stations: 0
set()


# SAVE STATION FEATURES

In [13]:
station_features.to_csv(f"{STATION_FEATURE_FOLDER}/station_features.csv", index=False)
print("Saved station_features.csv")

Saved station_features.csv
